# Load Delta Table

In [0]:
df = spark.read.table("workspace.default.facilities2")

# Handle null values

In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import DoubleType, LongType, IntegerType

# Identify column types
double_cols  = [f.name for f in df.schema.fields if isinstance(f.dataType, DoubleType)]
integer_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (LongType, IntegerType))]
string_cols  = [f.name for f in df.schema.fields if f.name not in double_cols + integer_cols]

# Fill nulls by type — NEVER fill numeric cols with ""
df = df.fillna("", subset=string_cols)   # strings → empty string
df = df.fillna(0.0, subset=double_cols)  # lat/lon → 0.0  (stays as DOUBLE)

# Mark unknown for numberDoctors & capacity columns

In [0]:
df = df.withColumn(
    "numberDoctors",
    when(col("numberDoctors") == 0, "unknown")
    .otherwise(col("numberDoctors").cast("string"))
).withColumn(
    "capacity",
    when(col("capacity") == 0, "unknown")
    .otherwise(col("capacity").cast("string"))
)


# Fix formatting

In [0]:
# Fix JSON strings
from pyspark.sql.functions import col, regexp_replace

df = df.withColumn("specialties", regexp_replace(col("specialties"), "'", '"')) \
       .withColumn("procedure", regexp_replace(col("procedure"), "'", '"')) \
       .withColumn("equipment", regexp_replace(col("equipment"), "'", '"')) \
       .withColumn("capability", regexp_replace(col("capability"), "'", '"'))

# Parse JSON → ARRAY
from pyspark.sql.functions import from_json
from pyspark.sql.types import ArrayType, StringType

array_schema = ArrayType(StringType())

df = df.withColumn("specialties", from_json(col("specialties"), array_schema)) \
       .withColumn("procedure", from_json(col("procedure"), array_schema)) \
       .withColumn("equipment", from_json(col("equipment"), array_schema)) \
       .withColumn("capability", from_json(col("capability"), array_schema))
# Remove null / empty values
from pyspark.sql.functions import expr

df = df.withColumn(
    "specialties",
    expr("filter(specialties, x -> x is not null and x != '')")
).withColumn(
    "procedure",
    expr("filter(procedure, x -> x is not null and x != '')")
).withColumn(
    "equipment",
    expr("filter(equipment, x -> x is not null and x != '')")
).withColumn(
    "capability",
    expr("filter(capability, x -> x is not null and x != '')")
)

# Convert to clean bullet text
from pyspark.sql.functions import when, size, concat_ws

df = df.withColumn(
    "specialties_text",
    when(size(col("specialties")) > 0,
         concat_ws("\n- ", col("specialties")))
    .otherwise("none")
)

df = df.withColumn(
    "procedure_text",
    when(size(col("procedure")) > 0,
         concat_ws("\n- ", col("procedure")))
    .otherwise("none")
).withColumn(
    "equipment_text",
    when(size(col("equipment")) > 0,
         concat_ws("\n- ", col("equipment")))
    .otherwise("none")
).withColumn(
    "capability_text",
    when(size(col("capability")) > 0,
         concat_ws("\n- ", col("capability")))
    .otherwise("none")
)


# Build Document

In [0]:
from pyspark.sql.functions import concat, lit

df_docs = df.withColumn(
    "document",
    concat(
        lit("[FACILITY]\n"),
        lit("Organization Type: "), col("organization_type"), lit("\n"),
        lit("Name: "), col("name"), lit("\n"),
        lit("Facility Type: "), col("facilityTypeId"), lit("\n\n"),

        lit("[LOCATION]\n"),
        lit("Location: "), col("osm_display_name"), lit("\n\n"),

        lit("[RESOURCES]\n"),
        lit("Number of Doctors: "), col("numberDoctors"), lit("\n"),
        lit("Patient Bed Capacity: "), col("capacity"), lit("\n\n"),

        lit("[MEDICAL DATA]\n"),
        lit("Specialties:\n- "), col("specialties_text"), lit("\n\n"),
        lit("Capabilities:\n- "), col("capability_text"), lit("\n\n"),
        lit("Procedures:\n- "), col("procedure_text"), lit("\n\n"),
        lit("Equipment:\n- "), col("equipment_text"), lit("\n\n"),

        lit("[DESCRIPTION]\n"),
        col("description")
    )
)

# Add metadata as columns

In [0]:
df_docs = df_docs.select(
    col("pk_unique_id").alias("id"),
    "document",
    "name",
    "organization_type",
    col("facilityTypeId").alias("facility_type"),
    "address_city",
    "address_stateOrRegion",
    "address_country",
    "lat",
    "lon",
    "numberDoctors",
    "capacity"
)

# Save delta table

In [0]:
df_docs.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("facility_documents")

# Preview

In [0]:
display(df_docs.select("name", "document").limit(5))